In [ ]:
import data
import device_types as dt

data_conn = data.connect()
dt_conn = dt.connect()

In [ ]:


for f in dt.(dt_conn):
  print(dict(f))


# Challenge 1

In [ ]:
conn = data.connect()

In [ ]:
from collections import Counter

def norm(value: str | None) -> str:
    return " ".join((value or "").lower().split())

In [ ]:
for f in conn.execute("SELECT id, record_count FROM facilities ORDER BY id").fetchall():
  records = conn.execute("SELECT * from records WHERE facility_id = ?", (f["id"])).fetchall()
  print(f"True record count {f["id"]}" if len(records) == f["record_count"] else False)

keys = Counter((norm(r["manufacturer"]), norm(r["model_name"])) for r in records)
print(f"True keys match total records" if sum(keys.values()) == len(records) else False)

In [ ]:
# sum of all blank records
sum(1 for r in records if not norm(r["manufacturer"]) or not norm(r["model_name"]))

In [ ]:
default_pairs = {(r["manufacturer"], r["model_name"]) for r in records}

In [ ]:
default_pairs

In [ ]:
# check if default pairs equals norm'd pairs
len(default_pairs) == len(keys)

In [ ]:
# singular entity pair
sum(1 for n in keys.values() if n == 1)

In [ ]:
top = keys.most_common(2)

In [ ]:
top

In [ ]:
# checking for records with duplicate CEID
duped_ceid = conn.execute("SELECT ceid, COUNT(*) n FROM records GROUP BY ceid HAVING n > 1").fetchall()
dict(duped_ceid)

In [ ]:

for d in duped_ceid:
  records = conn.execute(f"SELECT * FROM records WHERE CEID IS {d[0]}")
  for r in records:
    print(f"{dict(r)} \n")

In [ ]:
# checking for entire duplicate records
entire_duped = conn.execute("SELECT COUNT(*) FROM (SELECT 1 FROM records GROUP BY facility_id, ceid, serial_number, manufacturer, model_name, department HAVING COUNT(*) > 1)").fetchone()[0]
# for r in entire_duped:
#   print(dict(entire_duped))
entire_duped

# Challenge 2

In [ ]:
medc = ["A", "B", "C", "D"]


In [ ]:
results = []

In [ ]:
for cust in medc:
  results.append(conn.execute(f"SELECT customer, manufacturer, model_name, COUNT(*) as total_count FROM records WHERE customer = 'Medical Center {cust}' GROUP BY customer, manufacturer, model_name ORDER BY customer, total_count DESC"))

In [ ]:
for f in results:
  for fac in f:
    print(dict(fac))

In [ ]:
counts = {}
for fac in medc:
  rows = conn.execute("SELECT lower(trim(manufacturer)) as ma, lower(trim(model_name)) as mn, COUNT(*) AS n FROM records WHERE facility_id = ? GROUP BY manufacturer, model_name", (fac)).fetchall()
  counts[fac] = {(r["ma"], r["mn"]): r["n"] for r in rows}
print({fac: len(counts[fac]) for fac in medc})
print(counts)

In [ ]:
counts = {}
for fac in medc:
  rows = conn.execute("SELECT manufacturer, model_name, COUNT(*) AS n FROM records WHERE facility_id = ? GROUP BY manufacturer, model_name", (fac)).fetchall()
  counts[fac] = {(r["manufacturer"], r["model_name"]): r["n"] for r in rows}
print({fac: len(counts[fac]) for fac in medc})
print(counts)


In [ ]:
shared = set(counts["A"])
for fac in ["B", "C", "D"]:
  shared = shared & set(counts[fac])
print(len(shared))

In [ ]:
models = []

for manufacturer, model in shared:
  per_facility = {}
  for fac in medc:
    per_facility[fac] = counts[fac][(manufacturer, model)]
  models.append({
    "manufacturer": manufacturer,
    "model": model,
    "total": sum(per_facility.values()),
    "per_facility": per_facility
  })

In [ ]:
models

In [ ]:
data_conn.close()

# Challenge 3

## trying to cluster using similarity and containment to get clusters

In [ ]:
rows = data_conn.execute("SELECT DISTINCT manufacturer FROM records ORDER BY manufacturer ASC")
for row in rows:
  print(dict(row))

In [ ]:
import re
from collections import Counter, defaultdict
def tokens(text):
    return re.sub(r"[^a-z0-9 ]", " ", (text or "").lower().replace(".", "")).split()

In [ ]:
conn = data.connect()
rows = conn.execute(
    "SELECT manufacturer AS raw, COUNT(*) AS records, group_concat(DISTINCT facility_id) AS sources FROM records GROUP BY manufacturer"
).fetchall()
conn.close()


In [ ]:
by_raw = {r["raw"]: r for r in rows}
names = {raw: tokens(raw) for raw in by_raw}

In [ ]:
THRESHOLD = 0.85
GENERIC_IF_IN_AT_LEAST = 10

In [ ]:
token_freq = Counter(t for toks in names.values() for t in set(toks))
generic = {t for t, n in token_freq.items() if n >= GENERIC_IF_IN_AT_LEAST}
print(sorted(generic))


In [ ]:
from difflib import SequenceMatcher
def similarity(a, b):
    ta, tb = names[a], names[b]
    ratio = SequenceMatcher(None, " ".join(ta), " ".join(tb)).ratio()
    ka = {t for t in ta if t not in generic}
    kb = {t for t in tb if t not in generic}
    shared = ka & kb
    containment = len(shared) / min(len(ka), len(kb)) if shared else 0
    return ratio#max(ratio, containment)

def find(x):
  while parent[x] != x:
      parent[x] = parent[parent[x]]
      x = parent[x]
  return x

In [ ]:
parent = {raw: raw for raw in names}


In [ ]:
from itertools import combinations


best = defaultdict(float)
for a, b in combinations(names, 2):
  score = similarity(a, b)
  if score >= THRESHOLD:
    parent[find(a)] = find(b)
    best[a] = max(best[a], score)
    best[b] = max(best[b], score)


In [ ]:
best

In [ ]:
parent

## trying to get first token and cluster based on that, since first token is mostly unique to every manufacturer

In [90]:
import csv
import re
from collections import Counter
import data


def squash(name):
    return re.sub(r"[^a-z0-9]", "", (name or "").lower())


def first_word(name):
    words = re.sub(r"[^a-z0-9 ]", " ", (name or "").lower().replace(".", "")).split()
    if len(words) > 1 and words[0] == "the":
        words = words[1:]
    return words[0] if words else ""


conn = data.connect()
rows = conn.execute(
    "SELECT manufacturer AS raw, COUNT(*) AS records, group_concat(DISTINCT facility_id) AS sources "
    "FROM records GROUP BY manufacturer"
).fetchall()
conn.close()

out = [
    {"group": first_word(r["raw"]), "raw_manufacturer": r["raw"], "squashed": squash(r["raw"]),
     "records": r["records"], "sources": r["sources"]}
    for r in rows
]
sizes = Counter(d["group"] for d in out)
for d in out:
    d["group_size"] = sizes[d["group"]]
out.sort(key=lambda d: (d["group"], d["squashed"]))

with open("cache/step1_manufacturer_groups.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(out[0]))
    writer.writeheader()
    writer.writerows(out)

print(len(out), "manufacturers ->", len(sizes), "groups")
print(sizes.most_common(15))

708 manufacturers -> 603 groups
[('american', 8), ('medtronic', 6), ('ge', 5), ('medical', 5), ('siemens', 5), ('reichert', 4), ('zoll', 4), ('3m', 3), ('advanced', 3), ('aerogen', 3), ('baxter', 3), ('exergen', 3), ('medline', 3), ('mortara', 3), ('olympus', 3)]


In [91]:
sizes

Counter({'american': 8,
         'medtronic': 6,
         'ge': 5,
         'medical': 5,
         'siemens': 5,
         'reichert': 4,
         'zoll': 4,
         '3m': 3,
         'advanced': 3,
         'aerogen': 3,
         'baxter': 3,
         'exergen': 3,
         'medline': 3,
         'mortara': 3,
         'olympus': 3,
         'philips': 3,
         'samsung': 3,
         'smith': 3,
         'steris': 3,
         'stryker': 3,
         'united': 3,
         'welch': 3,
         'a': 2,
         'abbott': 2,
         'amsco': 2,
         'arjo': 2,
         'bio': 2,
         'cardinal': 2,
         'carefusion': 2,
         'carl': 2,
         'conmed': 2,
         'covidien': 2,
         'datex': 2,
         'drucker': 2,
         'fisher': 2,
         'fluke': 2,
         'fujifilm': 2,
         'general': 2,
         'global': 2,
         'hamilton': 2,
         'health': 2,
         'hill': 2,
         'hillrom': 2,
         'international': 2,
         'iris': 2,


In [92]:
out

[{'group': '3gen',
  'raw_manufacturer': '3GEN',
  'squashed': '3gen',
  'records': 5,
  'sources': 'A',
  'group_size': 1},
 {'group': '3m',
  'raw_manufacturer': '3M COMPANY',
  'squashed': '3mcompany',
  'records': 3,
  'sources': 'P',
  'group_size': 3},
 {'group': '3m',
  'raw_manufacturer': '3M HEALTHCARE',
  'squashed': '3mhealthcare',
  'records': 75,
  'sources': 'A,C,D,P',
  'group_size': 3},
 {'group': '3m',
  'raw_manufacturer': '3M MEDICAL',
  'squashed': '3mmedical',
  'records': 3,
  'sources': 'P',
  'group_size': 3},
 {'group': 'a',
  'raw_manufacturer': 'A-DEC',
  'squashed': 'adec',
  'records': 4,
  'sources': 'A',
  'group_size': 2},
 {'group': 'a',
  'raw_manufacturer': 'A-M SYSTEMS',
  'squashed': 'amsystems',
  'records': 6,
  'sources': 'A,B',
  'group_size': 2},
 {'group': 'aaron',
  'raw_manufacturer': 'AARON MEDICAL INDUSTRIES',
  'squashed': 'aaronmedicalindustries',
  'records': 3,
  'sources': 'B,E',
  'group_size': 1},
 {'group': 'abbott',
  'raw_manufac

## Trying a reverse cluster based on model name

In [ ]:
MIN_LEN = 4

conn = data.connect()
rows = conn.execute(
    "SELECT manufacturer, model_name, COUNT(*) AS n FROM records GROUP BY manufacturer, model_name"
).fetchall()
conn.close()

by_model = defaultdict(Counter)
example = {}
for r in rows:
    key = squash(r["model_name"])
    by_model[key][r["manufacturer"]] += r["n"]
    example.setdefault(key, r["model_name"])

cross = {
    key: mfrs
    for key, mfrs in by_model.items()
    if len(key) >= MIN_LEN and len(mfrs) > 1
}
print(len(by_model), "models |", len(cross), "span more than one first-word group")

out = [
    {
        "model": example[key],
        "manufacturer_count": len(mfrs),
        "manufacturers": " | ".join(f"{m} ({n})" for m, n in mfrs.most_common()),
        "records": sum(mfrs.values()),
    }
    for key, mfrs in cross.items()
]
out.sort(key=lambda d: (d["manufacturer_count"], -d["records"]))

with open("cache/step2_model_bridge.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(out[0]))
    writer.writeheader()
    writer.writerows(out)

for d in out[:40]:
    print(d["model"], "->", d["manufacturers"])

2052 models | 119 span more than one first-word group
MEDFUSION 3500 -> SMITHS MEDICAL (44) | SMITHS MEDICAL ASD (2)
CONCHATHERM NEPTUNE -> HUDSON RESPIRATORY CARE (28) | TELEFLEX MEDICAL (1)
V60 NON-INVASIVE -> PHILIPS MEDICAL SYSTEMS (26) | RESPIRONICS (2)
STATSPIN EXPRESS 4 -> IRIS SAMPLE PROCESSING (22) | BECKMAN COULTER (1)
LIFEPAK CR PLUS -> MEDTRONIC (16) | PHYSIO CONTROL (5)
152-001 -> MIDMARK (9) | RITTER (6)
PORTABLE LAMP -> UNDERWRITERS LABORATORIES (8) | ATLAS SPECIALTY LIGHTING (2)
GIRAFFE SPOT PT LITE -> OHMEDA (8) | DATEX OHMEDA (1)
OXIMAX N-65 -> NELLCOR PURITAN (7) | NELLCOR PURITAN BENNETT (2)
3 LITER SYRINGE -> A-M SYSTEMS (6) | MEDICAL GRAPHICS (1)
ECLIPSE -> NIKON (5) | INTERACOUSTICS (2)
AIDA -> STORZ (6) | KARL STORZ ENDOVISION (1)
LIFEPAK 1000 -> MEDTRONIC (6) | PHYSIO CONTROL (1)
ELITE 200 -> IMEX MEDICAL SYSTEMS (4) | NICOLET VASCULAR (2)
GIRAFFE WARMER -> GE MEDICAL SYSTEMS (3) | DATEX OHMEDA (2)
2500 -> FLUID FLO (4) | HEALTH O METER (1)
RC60XI/5-2 MHZ -> SO

## From this point on i am generating csv so that i can create an alias map of spellings to actual manufacturers. 

In [1]:
import csv
import json
import re
from collections import defaultdict
import data


def first_word(name):
    words = re.sub(r"[^a-z0-9 ]", " ", (name or "").lower().replace(".", "")).split()
    if len(words) > 1 and words[0] == "the":
        words = words[1:]
    return words[0] if words else ""


conn = data.connect()
rows = conn.execute("SELECT manufacturer, COUNT(*) AS n FROM records GROUP BY manufacturer").fetchall()
conn.close()

aliases = json.load(open("cache/alias-map.json"))["aliases"]
covered = {s for v in aliases.values() for s in v}

groups = defaultdict(list)
for r in rows:
    groups[first_word(r["manufacturer"])].append((r["manufacturer"], r["n"]))

out = []
for word, members in groups.items():
    if len(members) < 2 or all(m in covered for m, _ in members):
        continue
    members.sort(key=lambda x: -x[1])
    out.append({
        "model": f"[{word}]",
        "manufacturer_count": len(members),
        "manufacturers": " | ".join(f"{m} ({n})" for m, n in members),
        "records": sum(n for _, n in members),
    })
out.sort(key=lambda d: (d["manufacturer_count"], d["model"]))

with open("cache/step3_name_groups.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(out[0]))
    writer.writeheader()
    writer.writerows(out)

print(len(out), "groups")
print(sum(1 for d in out if d["manufacturer_count"] > 8), "groups with more than 8 names")

41 groups
0 groups with more than 8 names


In [2]:
import csv
import json
import data

aliases = json.load(open("cache/alias-map-uncovered.json"))["aliases"]
covered = {s for v in aliases.values() for s in v}

conn = data.connect()
rows = conn.execute("SELECT manufacturer, COUNT(*) AS n FROM records GROUP BY manufacturer ORDER BY n DESC").fetchall()
conn.close()

left = [(r["manufacturer"], r["n"]) for r in rows if r["manufacturer"] not in covered]
with open("cache/step4_leftovers.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["model", "manufacturer_count", "manufacturers", "records"])
    for i in range(0, len(left), 8):
        chunk = left[i:i + 8]
        writer.writerow([f"[leftovers {i // 8 + 1}]", len(chunk), " | ".join(f"{m} ({n})" for m, n in chunk), sum(n for _, n in chunk)])
print(len(left))

478


In [3]:
import csv
import json
import re
from collections import Counter, defaultdict
from difflib import SequenceMatcher
import data

aliases = json.load(open("cache/final_alias_map.json"))["aliases"]
to_canon = {s: c for c, v in aliases.items() for s in v}

conn = data.connect()
rows = conn.execute(
    "SELECT manufacturer, model_name, COUNT(*) AS n FROM records GROUP BY manufacturer, model_name"
).fetchall()
conn.close()

models = defaultdict(Counter)
for r in rows:
    canon = to_canon.get(r["manufacturer"], r["manufacturer"])
    models[canon][r["model_name"]] += r["n"]

squash = lambda s: re.sub(r"[^a-z0-9]", "", s.lower())
digits = lambda s: re.sub(r"\D", "", s)

out = []
for canon, spellings in models.items():
    by_key = defaultdict(list)
    for sp in spellings:
        by_key[squash(sp)].append(sp)

    involved = {k for k, v in by_key.items() if len(v) > 1}
    keys = list(by_key)
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            a, b = keys[i], keys[j]
            if min(len(a), len(b)) >= 8 and digits(a) == digits(b) and SequenceMatcher(None, a, b).ratio() >= 0.9:
                involved.update((a, b))

    chips = [sp for k in involved for sp in by_key[k]]
    if not chips:
        continue
    chips.sort(key=lambda sp: -spellings[sp])
    out.append({
        "model": canon,
        "manufacturer_count": len(chips),
        "manufacturers": " | ".join(f"{canon}||{sp} ({spellings[sp]})" for sp in chips),
        "records": sum(spellings[sp] for sp in chips),
    })
out.sort(key=lambda d: -d["records"])

with open("cache/step5_model_candidates.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(out[0]))
    writer.writeheader()
    writer.writerows(out)

with open("cache/step5_model_candidates.csv") as f:
    csv_text = f.read()
progress = {"csv": csv_text, "sample": False, "buckets": {}, "assign": {}, "status": {}, "cur": None, "nextId": 1}
with open("cache/step5_model_progress.json", "w") as f:
    json.dump(progress, f)

print(len(out), "manufacturers with candidates |", sum(d["manufacturer_count"] for d in out), "model spellings")

25 manufacturers with candidates | 95 model spellings


In [4]:
import json
from collections import defaultdict
import data

mfr_map = {s: c for c, v in json.load(open("cache/final_alias_map.json"))["aliases"].items() for s in v}

conn = data.connect()
rows = conn.execute(
    "SELECT facility_id, manufacturer, model_name, COUNT(*) AS n FROM records GROUP BY facility_id, manufacturer, model_name"
).fetchall()
conn.close()

sources = defaultdict(lambda: defaultdict(int))
for r in rows:
    sources[r["manufacturer"]][r["facility_id"]] += r["n"]
    sources[f"{mfr_map.get(r['manufacturer'], r['manufacturer'])}||{r['model_name']}"][r["facility_id"]] += r["n"]

with open("cache/spelling_sources.json", "w") as f:
    json.dump({"sources": sources}, f)
print(len(sources), "spellings")

2814 spellings
